# Lexical and structural repository retrieval

Goal: find exact code words, identifiers, paths, and symbols without an embedding model or chat LLM. This is a separate retrieval layer that will later be combined with semantic search.

This tutorial is standalone. On a new computer, first run the setup command in the README and select QAChat's `.venv` kernel.

In [2]:
from pathlib import Path

root = Path.cwd()
if not (root / "qachat").is_dir():
    root = root.parent

print(f"QAChat root: {root}")

QAChat root: /Users/yuwei.chang/OnDevice/GitHub/QAChat


In [3]:
import sys

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

print(f"Python environment: {sys.executable}")

Python environment: /Users/yuwei.chang/OnDevice/GitHub/QAChat/.venv/bin/python


## 1. Prepare chunks

A visible word counter is enough here because lexical and structural search do not use vectors. The same shared source and chunk code is used by the application.

In [4]:
from qachat.chunking import chunk_source_units
from qachat.repository import scan_repository
from qachat.source import extract_source_units, read_repository_files

report = scan_repository(
    root,
    extensions=[".py", ".md"],
    include_paths=["qachat", "README.md"],
)
source_files = read_repository_files(report)
units = extract_source_units(source_files)
chunks = chunk_source_units(
    units,
    count_tokens=lambda text: len(text.split()),
    target_tokens=120,
    max_tokens=160,
    overlap_tokens=20,
)

print(f"Prepared {len(chunks)} chunks.")

Prepared 98 chunks.


## 2. Search exact source text

Lexical search looks inside chunk text. It keeps a complete identifier and also splits snake case and camel case. Therefore `EmbeddingSession`, `embedding_session`, and the words `embedding session` can share searchable terms.

In [5]:
from qachat.retrieval import lexical_search

LEXICAL_QUESTION = "embedding session"
lexical_results = lexical_search(
    chunks,
    LEXICAL_QUESTION,
    count=5,
)

for result in lexical_results:
    chunk = result.chunk
    location = f"{chunk.path}:{chunk.start_line}-{chunk.end_line}"
    print(f"{result.score:.3f}  {location}  {chunk.symbol}")

4.838  qachat/embeddings.py:107-140  EmbeddingSession.count_tokens
4.469  qachat/retrieval.py:60-77  SemanticIndex
4.363  qachat/embeddings.py:142-179  EmbeddingSession.embed_documents
3.775  qachat/embeddings.py:32-67  EmbeddingSession
3.747  qachat/embeddings.py:1-31  embeddings


## 3. Search source metadata

Structural search looks at file paths, filenames, symbols, section names, and unit kinds instead of source text. It is useful for questions such as where a class or file is located.

In [6]:
from qachat.retrieval import structural_search

STRUCTURAL_QUESTION = "Where is EmbeddingSession defined?"
structural_results = structural_search(
    chunks,
    STRUCTURAL_QUESTION,
    count=5,
)

for result in structural_results:
    chunk = result.chunk
    location = f"{chunk.path}:{chunk.start_line}-{chunk.end_line}"
    print(f"{result.score:.1f}  {location}  {chunk.symbol}")

15.0  qachat/embeddings.py:255-284  EmbeddingSession._load_torch
15.0  qachat/embeddings.py:68-105  EmbeddingSession.__init__
15.0  qachat/embeddings.py:181-206  EmbeddingSession.embed_query
15.0  qachat/embeddings.py:286-319  EmbeddingSession._embed_mlx
15.0  qachat/embeddings.py:208-223  EmbeddingSession.close


## Result

The two result lists remain separate and explain which method produced each score. A later layer will combine semantic, lexical, and structural rankings without changing these functions.